### imports

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle as pk
import mlflow
import xgboost as xgb
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import root_mean_squared_error
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

!python -V

Python 3.12.7


### configs

In [2]:
# configs
# set the tracking uri to the local sqlite db
mlflow.set_tracking_uri("sqlite:///mlflow.db")
# set the experiment name
mlflow.set_experiment("NYC-taxi-duration-prediction")
# access to the ui
#$ mlflow ui --backend-store-uri sqlite:///mlflow.db

<Experiment: artifact_location='/home/yezergm/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1743783492519, experiment_id='1', last_update_time=1743783492519, lifecycle_stage='active', name='NYC-taxi-duration-prediction', tags={}>

### dev

In [ ]:
df = pd.read_parquet('./data/green_tripdata_2024-12.parquet')
df.passenger_count.value_counts()

### functions

In [3]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


### get X_train, y_train, X_val, y_val

In [6]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')
print(len(df_train), len(df_val))

df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

cat = ['PU_DO']
num = ['trip_distance']
target = 'duration'

dv = DictVectorizer()
train_dicts = df_train[cat + num].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
print(X_train.shape[1])
y_train = df_train[target].values
val_dicts = df_val[cat + num].to_dict(orient='records')
X_val = dv.transform(val_dicts)
y_val = df_val[target].values

73908 61921
13221


### linear regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

##### plots

In [ ]:
y_pred_train = model.predict(X_train)
print(f'RMSE train: {root_mean_squared_error(y_train, y_pred_train)}')
y_pred_val = model.predict(X_val)
print(f'RMSE val: {root_mean_squared_error(y_val, y_pred_val)}')

sns.histplot(y_pred_train, label='prediction')
sns.histplot(y_train, label='actual')
plt.legend()
plt.show()

##### saving the model

In [21]:
with open('./models/lin_reg.bin', 'wb') as f_out:
    pk.dump((model, dv), f_out)

### training Lasso & track mlflow

In [ ]:
with mlflow.start_run():

    alpha = 0.001
    mlflow.set_tag("developer", "yezer")
    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")
    mlflow.log_param("alpha", alpha)
    
    model = Lasso(alpha=alpha)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    print(f'RMSE train: {root_mean_squared_error(y_train, y_pred_train)}')
    y_pred_val = model.predict(X_val)

    with open('./models/lin_reg.bin', 'wb') as f_out:
        pk.dump((model, dv), f_out)

    print(f'RMSE val: {root_mean_squared_error(y_val, y_pred_val)}')
    mlflow.log_metric("rmse-train", root_mean_squared_error(y_train, y_pred_train))
    mlflow.log_metric("rmse-val", root_mean_squared_error(y_val, y_pred_val))

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle/")
    

### training XGBoost & mlflow

In [ ]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

##### best xgboost model

In [ ]:
params = {
    "learning_rate": 0.3989983259210908,
    "max_depth": 33,
    "min_child_weight": 2.029221938707499,
    "reg_alpha": 0.006990925176038303,
    "reg_lambda": 0.005689701992348848,
    "seed": 42,
}

# way 01
# with mlflow.start_run():
#     mlflow.set_tag("model", "xgboost")
#     mlflow.log_params(params)
#     .....
# or

# way 02
# mlflow.xgboost.autolog() #some frameworks have autolog() as xgboost, scikit-learn, etc.


# way 3
mlflow.xgboost.autolog(disable=True)
input_example = X_val[:1]  # Take first row as example
with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    booster = xgb.train(
        params=params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, "validation")],
        early_stopping_rounds=50,
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)

    with open('./models/preprocessor.b', 'wb') as f_out:
        pk.dump(dv, f_out)

    # way 03
    mlflow.log_params(params)
    mlflow.log_metric("rmse", rmse)
    mlflow.set_tag("model", "xgboost")
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")    
    # the best way to store the model is this, there are a lot of frameworks compatible with mlflow
    # Keras, Pytorch, ONXX, etc.
    mlflow.xgboost.log_model(
        booster, artifact_path="models_mlflow"
    )



##### Make predictions. Validate the model before deployment
###### Copied from mlflow ui

In [ ]:
import mlflow
import pandas as pd

logged_model = "runs:/3463b2d27a214a959b40ab09c3307d55/models_mlflow"

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)

# Predict
predictions = loaded_model.predict(X_val)
print(predictions[:10])
y_pred = booster.predict(valid)
print(y_pred[:10])

### five regressors & mlflow

##### Training and track

In [8]:

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR
from time import time

with open('./models/preprocessor.b', 'wb') as f_out:
    pk.dump(dv, f_out)

# whether enable, it tracks a lot of metrics.
mlflow.sklearn.autolog(disable=True)

# If I don't use this, the model will be trained with the default parameters, takes too long
model_configs = {
    GradientBoostingRegressor: {'n_estimators': 50, 'max_depth': 10},
    ExtraTreesRegressor: {'n_estimators': 50, 'max_depth': 20, 'n_jobs': -1},
    LinearSVR: {'max_iter': 2000, 'dual': False, 'loss': 'squared_epsilon_insensitive'},
    RandomForestRegressor: {'n_estimators': 50, 'max_depth': 20, 'n_jobs': -1}
}
for model_class in (GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR, RandomForestRegressor):

    with mlflow.start_run():
        print(f'Training {model_class.__name__}')
        mlflow.set_tag("developer", "yezer")
        mlflow.set_tag("model", model_class.__name__)
        mlflow.set_tags({"developer": "yezer", "model": model_class.__name__})
        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        # Log model configuration parameters
        mlflow.log_params(model_configs[model_class])

        mlmodel = model_class(**model_configs[model_class])
        t0 = time()
        mlmodel.fit(X_train, y_train)
        t_time = time() - t0
        print(f'Training time: {t_time}')
        mlflow.sklearn.log_model(
                mlmodel, artifact_path="models_mlflow"
            )

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

Training GradientBoostingRegressor
Training time: 22.760976552963257


2025/04/08 13:26:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training ExtraTreesRegressor
Training time: 19.184048175811768


2025/04/08 13:26:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training LinearSVR
Training time: 7.444507598876953


2025/04/08 13:26:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Training RandomForestRegressor
Training time: 24.518590211868286


2025/04/08 13:27:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
